In [1]:
# Hyperparameter tuning notebook
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [2]:
# Importing the necessary libraries and functions
import pandas as pd
from src.data_ingestion import load_application_train, load_bureau, load_previous_application
from src.feature_engineering import engineer_features_pipeline, prepare_model_data

pd.set_option('display.max_columns', None)

In [3]:
# Loading the data and splitting it into training and testing sets
df_raw = load_application_train()
bureau_raw = load_bureau()
prev_app_raw = load_previous_application()

df = engineer_features_pipeline(df_raw, bureau_raw, prev_app_raw)
X_train_scaled, X_test_scaled, y_train, y_test, scaler = prepare_model_data(df)

print(X_train_scaled.shape, X_test_scaled.shape)

Loaded application_train: 307511 rows, 122 columns
Loaded bureau: 1716428 rows, 17 columns
Loaded previous_application: 1670214 rows, 37 columns
(246008, 249) (61503, 249)


In [4]:
# Re-training XGBoost model
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

baseline_xgb = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
baseline_xgb.fit(X_train_scaled, y_train)

baseline_auc = roc_auc_score(y_test, baseline_xgb.predict_proba(X_test_scaled)[:, 1])
print(f"Baseline XGBoost ROC-AUC: {baseline_auc:.4f}")

Baseline XGBoost ROC-AUC: 0.7568


## Key XGBoost Hyperparameters

- **n_estimators**: Number of boosting rounds (trees). More trees can improve performance but risks overfitting and increases training time.
- **max_depth**: Maximum depth of each tree. Deeper trees capture more complex patterns but are more prone to overfitting.
- **learning_rate**: How much each tree contributes to the final prediction. Lower values need more trees but often generalize better 
(a common tradeoff with n_estimators).
- **subsample**: Fraction of training rows used for each tree. Values below 1.0 add randomness, which can reduce overfitting 
(similar idea to bagging in Random Forest).
- **colsample_bytree**: Fraction of features considered for each tree. Same reasoning as subsample, but for columns instead of rows.
- **min_child_weight**: Minimum sum of instance weight needed in a child node. Higher values make the model more conservative (less likely to create overly specific splits on noise).

In [5]:
# Set up RandomizedSearchCV for hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

xgb_for_search = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=xgb_for_search,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    verbose=1,
    n_jobs=1 
)

print("Search configured")

Search configured


In [6]:
# Running the configured search
random_search.fit(X_train_scaled, y_train)
print("Search complete")
print(f"Best CV ROC-AUC: {random_search.best_score_:.4f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Search complete
Best CV ROC-AUC: 0.7619


In [7]:
# Extracting and interpreting the best parameters
best_params = random_search.best_params_
print(best_params)

{'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


In [8]:
# Tuning the XGBoost object with the best parameters
tuned_xgb = XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

tuned_xgb.fit(X_train_scaled, y_train)

print("Tuned XGBoost trained successfully")

Tuned XGBoost trained successfully


In [9]:
# Comparing the performance of both models (Baseline & Tuned)
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_pred_tuned = tuned_xgb.predict(X_test_scaled)
y_pred_proba_tuned = tuned_xgb.predict_proba(X_test_scaled)[:, 1]

y_pred_baseline = baseline_xgb.predict(X_test_scaled)
y_pred_proba_baseline = baseline_xgb.predict_proba(X_test_scaled)[:, 1]

comparison = pd.DataFrame({
    'Metric': ['ROC-AUC', 'Recall', 'Precision', 'F1'],
    'Baseline XGBoost': [
        roc_auc_score(y_test, y_pred_proba_baseline),
        recall_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_baseline),
        f1_score(y_test, y_pred_baseline),
    ],
    'Tuned XGBoost': [
        roc_auc_score(y_test, y_pred_proba_tuned),
        recall_score(y_test, y_pred_tuned),
        precision_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned),
    ]
})

comparison['Improvement'] = comparison['Tuned XGBoost'] - comparison['Baseline XGBoost']
print(comparison.round(4))

      Metric  Baseline XGBoost  Tuned XGBoost  Improvement
0    ROC-AUC            0.7568         0.7667       0.0099
1     Recall            0.6201         0.6912       0.0711
2  Precision            0.1833         0.1724      -0.0109
3         F1            0.2829         0.2760      -0.0069
